# GPM IMERG monthly precipitation (NASA Earthdata)

Download a monthly GPM IMERG precipitation granule from GES DISC through one Earthdata Login, then map it. This is a **live** query: it needs the `[earthdata]` extra (Python >=3.12) and EDL credentials (`EARTHDATA_USERNAME` / `EARTHDATA_PASSWORD` or `~/.netrc`). The query is wrapped so the notebook stays safe under nbval-lax when run offline / without credentials. See [Authentication](../../reference/earthdata/authentication.md) and [Usage](../../reference/earthdata/usage.md).

## Setup

Import the unified `EarthLens` entry point and prepare a local output directory. `download()` writes the fetched granule(s) into `earthdata_output/`.

In [ ]:
from pathlib import Path

from earthlens.core import EarthLens

OUT_DIR = Path('earthdata_output')
OUT_DIR.mkdir(exist_ok=True)

## Download the IMERG granule

### Build the request

Configure an `EarthLens` request for the monthly GPM IMERG dataset (`GPM_3IMERGM_07`), one month of June 2023, over a near-global window. Keeping construction on its own line makes the parameters easy to read and tweak.

In [ ]:
imerg = EarthLens(
    data_source='earthdata',
    dataset='GPM_3IMERGM_07',
    variables=['precipitation'],
    start='2023-06-01',
    end='2023-06-30',
    aoi=[-180.0, -60.0, 180.0, 60.0],
    path=OUT_DIR,
)

### Fetch it

Run `download()` to fetch the granule(s) to disk. The call is wrapped in a `try`/`except` so the notebook degrades gracefully — printing a skip message instead of erroring — when run offline or without Earthdata credentials.

In [ ]:
paths = imerg.download(progress_bar=False)
print(len(paths), 'granule(s):', [Path(p).name for p in paths])

Open the granule with pyramids' `NetCDF` — IMERG keeps its fields under a single
HDF5 group, which `get_group` reaches — and plot the June 2023 mean precipitation
rate. The download above is no longer guarded: a failure there should stop the
notebook rather than be reported as a skipped step.

In [ ]:
from pyramids.netcdf import NetCDF
from pyramids.plot import Basemap, ColorBar, ColorScaling, Feature

nc = NetCDF.read_file(paths[0], read_only=True)
grid = nc.get_group('Grid')  # IMERG stores its fields under a single HDF5 group
precip = grid.get_variable('precipitation')
stats = precip.stats(approx_ok=False)
print(f'grid {precip.rows} x {precip.columns}, epsg {precip.epsg}')
print(
    f'rate min/mean/max: {float(stats["min"].iloc[0]):.2f} / '
    f'{float(stats["mean"].iloc[0]):.2f} / {float(stats["max"].iloc[0]):.2f} mm/hr'
)

precip.plot(
    cmap='YlGnBu',
    color=ColorScaling.power(gamma=0.4),
    # Coastlines only, no relief: the field is opaque and global, so a tile basemap
    # underneath it would never be seen. The outline is what makes the monsoons and
    # the subtropical dry zones readable as geography.
    basemap=Basemap(
        relief=False,
        features=[Feature('coastline', edgecolor='#444', linewidth=0.4)],
    ),
    colorbar=ColorBar(label='precipitation rate (mm/hr)'),
    title='GPM IMERG monthly precipitation rate, June 2023',
)
nc.close()